# LeetCode 154 — Find Minimum in Rotated Sorted Array II (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMin(self, nums: List[int]) -> int:

```

Nhìn qua, bài này có vẻ "dễ quá": chỉ cần `min(nums)` là xong! Đúng vậy, `min(nums)` **được chấp nhận** trên LeetCode. Nhưng đề bài nói rõ: *"You must decrease the overall operation steps as much as possible"* — hãy **giảm số bước** càng nhiều càng tốt. Cái hay (và cái khó) của bài Hard này nằm ở chỗ: mảng đã gần như sắp xếp, ta có thể dùng **binary search** (tìm kiếm nhị phân) để chỉ nhìn vài phần tử... nhưng **phần tử trùng nhau** lại phá hỏng một phần sức mạnh đó. Hiểu được *vì sao* bị phá hỏng và *sửa thế nào cho an toàn* chính là bài học chính.

---

## Sau notebook này, bạn sẽ nắm được

1. "Mảng sắp xếp bị xoay" (rotated sorted array) là gì, hình dạng "hai bậc thang" của nó, vẽ bằng ASCII.
2. **Cách 1 — Quét tuyến tính / `min()`** `O(n)`: cách ai cũng nghĩ ra, làm mốc để đối chiếu.
3. **Cách 2 — Tìm "điểm gãy"** `nums[i] > nums[i+1]`: lần đầu khai thác cấu trúc của mảng, có thể dừng sớm.
4. **Binary search từ căn bản**: trò chơi đoán số, `lo / hi / mid`, mẫu "tìm vị trí `True` đầu tiên".
5. **Cách 3 — Binary search cho bản KHÔNG trùng (LeetCode 153)** `O(log n)`: so `nums[mid]` với `nums[hi]`.
6. **Cách 4 — Binary search có xử lý trùng (cách chính)**: khi `nums[mid] == nums[hi]` thì `hi -= 1`. Vì sao bước này **an toàn**, và vì sao trường hợp xấu nhất là `O(n)` (ví dụ `[1,1,1,0,1,1]`).
7. **Cách 5 — Chia để trị bằng đệ quy** (tùy chọn): một góc nhìn khác về cùng ý tưởng.
8. Trả lời câu hỏi *Follow up* của đề, lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Suppose an array of length `n` sorted in ascending order is **rotated** between `1` and `n` times. For example, the array `nums = [0,1,4,4,5,6,7]` might become:

- `[4,5,6,7,0,1,4]` if it was rotated `4` times.
- `[0,1,4,4,5,6,7]` if it was rotated `7` times.

Notice that rotating an array `[a[0], a[1], a[2], ..., a[n-1]]` 1 time results in the array `[a[n-1], a[0], a[1], a[2], ..., a[n-2]]`.

Given the sorted rotated array `nums` that may contain **duplicates**, return *the minimum element of this array*.

You must decrease the overall operation steps as much as possible.

**Example 1**

```text
Input:  nums = [1,3,5]
Output: 1
```

**Example 2**

```text
Input:  nums = [2,2,2,0,1]
Output: 0
```

**Constraints**

- `n == nums.length`
- `1 <= n <= 5000`
- `-5000 <= nums[i] <= 5000`
- `nums` is sorted and rotated between `1` and `n` times.

**Follow up:** This problem is similar to *Find Minimum in Rotated Sorted Array* (LeetCode 153), but `nums` may contain duplicates. Would this affect the runtime complexity? How and why?

---

**Dịch nhanh và những điều cần để ý:**

- Mảng ban đầu được sắp xếp **tăng dần** (chính xác hơn là *không giảm*, vì có thể có số bằng nhau như `4, 4`), rồi bị **xoay** từ `1` tới `n` lần.
- "Xoay 1 lần" = lấy phần tử **cuối** đưa lên **đầu**.
- Xoay đúng `n` lần thì mảng quay về y như cũ (Example 1 `[1,3,5]` chính là trường hợp này) — đừng quên trường hợp "trông như chưa xoay"!
- Hàm trả về **giá trị** nhỏ nhất (một số nguyên), **không** phải chỉ số (index).
- Khác biệt duy nhất so với bài 153: ở đây **được phép có phần tử trùng**. Câu *Follow up* hỏi: điều đó có làm thuật toán chậm đi không? Cuối mục 11 ta sẽ trả lời đầy đủ.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

nums1 = [1, 3, 5]
nums2 = [2, 2, 2, 0, 1]

print("Example 1:", nums1, "-> mong đợi", 1)
print("Example 2:", nums2, "-> mong đợi", 0)
print("min() của Python cho ra:", min(nums1), "và", min(nums2))

Example 1: [1, 3, 5] -> mong đợi 1
Example 2: [2, 2, 2, 0, 1] -> mong đợi 0
min() của Python cho ra: 1 và 0


## 2. Nền tảng: "xoay mảng" (rotate) là gì?

Đề định nghĩa: xoay `[a0, a1, ..., a(n-1)]` **một lần** cho ra `[a(n-1), a0, a1, ..., a(n-2)]`. Nói cách khác: **phần tử cuối nhảy lên đầu**, mọi phần tử khác lùi sang phải một ô.

Ví dụ với `[0,1,4,4,5,6,7]`:

| số lần xoay | mảng thu được |
|---:|---|
| 1 | `[7,0,1,4,4,5,6]` |
| 2 | `[6,7,0,1,4,4,5]` |
| 4 | `[4,5,6,7,0,1,4]` (giống ví dụ của đề) |
| 7 | `[0,1,4,4,5,6,7]` (quay về như cũ) |

### Kiến thức Python: slicing `a[-1:]`, `a[:-1]`, `a[-k:]`

- `a[-1]` là phần tử **cuối**; `a[-1:]` là **list** chỉ chứa phần tử cuối.
- `a[:-1]` là mọi phần tử **trừ** phần tử cuối.
- Vậy xoay một lần: `[a[-1]] + a[:-1]` (dấu `+` giữa hai list là **nối** list).
- Xoay `k` lần liên tiếp = đưa `k` phần tử cuối lên đầu: `a[-k:] + a[:-k]`. (Với `k = n`: `a[-n:]` là cả mảng, `a[:-n]` là rỗng → mảng y như cũ.)

In [2]:
def rotate_once(a):
    """Xoay 1 lần theo đúng định nghĩa của đề: phần tử cuối lên đầu."""
    return [a[-1]] + a[:-1]


def rotate(a, k):
    """Xoay k lần (1 <= k <= n): đưa k phần tử cuối lên đầu."""
    return a[-k:] + a[:-k]


base = [0, 1, 4, 4, 5, 6, 7]
cur = base
for k in range(1, len(base) + 1):
    cur = rotate_once(cur)
    print(f"xoay {k} lần: {cur}   | rotate(base, {k}) = {rotate(base, k)}   khớp: {cur == rotate(base, k)}")

xoay 1 lần: [7, 0, 1, 4, 4, 5, 6]   | rotate(base, 1) = [7, 0, 1, 4, 4, 5, 6]   khớp: True
xoay 2 lần: [6, 7, 0, 1, 4, 4, 5]   | rotate(base, 2) = [6, 7, 0, 1, 4, 4, 5]   khớp: True
xoay 3 lần: [5, 6, 7, 0, 1, 4, 4]   | rotate(base, 3) = [5, 6, 7, 0, 1, 4, 4]   khớp: True
xoay 4 lần: [4, 5, 6, 7, 0, 1, 4]   | rotate(base, 4) = [4, 5, 6, 7, 0, 1, 4]   khớp: True
xoay 5 lần: [4, 4, 5, 6, 7, 0, 1]   | rotate(base, 5) = [4, 4, 5, 6, 7, 0, 1]   khớp: True
xoay 6 lần: [1, 4, 4, 5, 6, 7, 0]   | rotate(base, 6) = [1, 4, 4, 5, 6, 7, 0]   khớp: True
xoay 7 lần: [0, 1, 4, 4, 5, 6, 7]   | rotate(base, 7) = [0, 1, 4, 4, 5, 6, 7]   khớp: True


## 3. Hình dạng của mảng đã xoay: "hai bậc thang"

Xoay `k` lần nghĩa là cắt mảng đã sắp xếp thành **hai khúc** rồi đổi chỗ: khúc cuối (dài `k`) lên trước. Ví dụ `base = [0,1,4,4,5,6,7]`, xoay 4 lần:

```text
base đã sắp xếp:   [0, 1, 4 | 4, 5, 6, 7]
                    khúc đầu   khúc cuối (4 phần tử)
sau khi xoay:      [4, 5, 6, 7 | 0, 1, 4]
                    phần A       phần B
```

Vẽ chiều cao từng phần tử, ta thấy **hai bậc thang** đi lên, giữa chúng có một chỗ **"gãy"** (rơi xuống):

```text
 giá trị
   7 |          #
   6 |       #  #
   5 |    #  #  #
   4 | #  #  #  #           #
   3 | #  #  #  #           #
   2 | #  #  #  #           #
   1 | #  #  #  #        #  #
   0 | #  #  #  #     #  #  #
     +----------------------
  i:   0  1  2  3  |  4  5  6
       phần A (cao) ↓ phần B (thấp)
                  chỗ gãy
```

Ba quan sát **cực kỳ quan trọng** cho cả bài:

1. Mỗi phần (A và B) là một đoạn **không giảm**.
2. **Mọi** phần tử của A đều `>=` **mọi** phần tử của B (vì A là khúc *cuối* của mảng sắp xếp, B là khúc *đầu*).
3. **Phần tử nhỏ nhất là phần tử đầu tiên của phần B** — tức là ngay sau chỗ gãy. Nếu mảng xoay đúng `n` lần thì không có chỗ gãy, cả mảng là một bậc thang và min là `nums[0]`.

Cell dưới là một hàm vẽ nhỏ để ta "nhìn" mảng. Bạn không cần hiểu hết chi tiết hàm vẽ; chỉ cần biết nó vẽ mỗi phần tử thành một cột `#`, cột càng cao thì giá trị càng lớn.

In [3]:
def draw_bars(nums, marks=None):
    """Vẽ mỗi phần tử thành một cột '#'. marks: dict {index: ký tự} để đánh dấu bên dưới."""
    low, top = min(nums), max(nums)
    for level in range(top, low - 1, -1):          # vẽ từ trên xuống
        row = " ".join("#" if v >= level else " " for v in nums)
        print(f"{level:>3} | {row}")
    print("    +-" + "-" * (2 * len(nums) - 1))
    print("  i:  " + " ".join(str(i % 10) for i in range(len(nums))))
    if marks:
        print("      " + " ".join(marks.get(i, " ") for i in range(len(nums))))


draw_bars([4, 5, 6, 7, 0, 1, 4], marks={4: "^"})
print("dấu ^ = vị trí min (đầu phần B)")

  7 |       #      
  6 |     # #      
  5 |   # # #      
  4 | # # # #     #
  3 | # # # #     #
  2 | # # # #     #
  1 | # # # #   # #
  0 | # # # # # # #
    +--------------
  i:  0 1 2 3 4 5 6
              ^    
dấu ^ = vị trí min (đầu phần B)


Với **phần tử trùng**, hình dạng vẫn là hai bậc thang, chỉ là có những "bậc phẳng". Example 2 `[2,2,2,0,1]`: phần A = `[2,2,2]` (phẳng), phần B = `[0,1]`.

Trường hợp đặc biệt khó chịu: hai đầu mảng **bằng nhau**, như `[1,1,1,0,1,1]`. Phần A = `[1,1,1]`, phần B = `[0,1,1]`. Nhìn hai đầu (đều là `1`), bạn **không đoán được** chỗ gãy ở đâu. Hãy nhớ hình này — nó là "kẻ phá đám" ở mục 9 và 11.

In [4]:
draw_bars([2, 2, 2, 0, 1], marks={3: "^"})
print()
draw_bars([1, 1, 1, 0, 1, 1], marks={3: "^"})

  2 | # # #    
  1 | # # #   #
  0 | # # # # #
    +----------
  i:  0 1 2 3 4
            ^  

  1 | # # #   # #
  0 | # # # # # #
    +------------
  i:  0 1 2 3 4 5
            ^    


## 4. Kiến thức Python dùng ngay: `min()`, index âm, `range(len(nums) - 1)`

- `min(nums)` duyệt **toàn bộ** list và trả về giá trị nhỏ nhất → luôn tốn `n` bước đọc.
- `nums[-1]` là phần tử cuối, `nums[-2]` là phần tử áp chót.
- Khi cần so sánh **mỗi phần tử với phần tử ngay sau nó** (`nums[i]` với `nums[i+1]`), ta cho `i` chạy `range(len(nums) - 1)`, tức `0, 1, ..., n-2`. Nếu cho `i` chạy tới `n-1` thì `nums[i+1]` sẽ **vượt chỉ số** (`IndexError`).

In [5]:
nums = [4, 5, 6, 7, 0, 1, 4]
print("min(nums) =", min(nums))
print("nums[-1] =", nums[-1], "| nums[-2] =", nums[-2])

for i in range(len(nums) - 1):
    print(f"i={i}: so nums[{i}]={nums[i]} với nums[{i+1}]={nums[i+1]}  ->  giảm? {nums[i] > nums[i+1]}")

min(nums) = 0
nums[-1] = 4 | nums[-2] = 1
i=0: so nums[0]=4 với nums[1]=5  ->  giảm? False
i=1: so nums[1]=5 với nums[2]=6  ->  giảm? False
i=2: so nums[2]=6 với nums[3]=7  ->  giảm? False
i=3: so nums[3]=7 với nums[4]=0  ->  giảm? True
i=4: so nums[4]=0 với nums[5]=1  ->  giảm? False
i=5: so nums[5]=1 với nums[6]=4  ->  giảm? False


## 5. Cách 1 — Quét tuyến tính (linear scan) / `min()`

**Ý tưởng tự nhiên nhất:** đi qua từng phần tử, nhớ giá trị nhỏ nhất đã thấy (`best`). Gặp số nhỏ hơn `best` thì cập nhật. Đây chính là những gì `min()` làm bên trong.

Cách này **không quan tâm** mảng có xoay, có sắp xếp hay có trùng hay không — nó đúng với **mọi** list. Ta sẽ dùng nó làm **mốc chuẩn** (baseline) để kiểm tra các cách thông minh hơn.

In [6]:
def find_min_builtin(nums: List[int]) -> int:
    return min(nums)


def find_min_linear(nums: List[int]) -> int:
    best = nums[0]
    for x in nums[1:]:
        if x < best:
            best = x
    return best


for nums in ([1, 3, 5], [2, 2, 2, 0, 1], [4, 5, 6, 7, 0, 1, 4], [1, 1, 1, 0, 1, 1], [7]):
    print(nums, "-> builtin:", find_min_builtin(nums), "| linear:", find_min_linear(nums))

[1, 3, 5] -> builtin: 1 | linear: 1
[2, 2, 2, 0, 1] -> builtin: 0 | linear: 0
[4, 5, 6, 7, 0, 1, 4] -> builtin: 0 | linear: 0
[1, 1, 1, 0, 1, 1] -> builtin: 0 | linear: 0
[7] -> builtin: 7 | linear: 7


### Trace cách 1 trên Example 2 `[2,2,2,0,1]`

| i | nums[i] | best trước | so sánh | best sau |
|---:|---:|---:|---|---:|
| 0 | 2 | — | khởi tạo | 2 |
| 1 | 2 | 2 | `2 < 2`? Không | 2 |
| 2 | 2 | 2 | `2 < 2`? Không | 2 |
| 3 | 0 | 2 | `0 < 2`? **Có** | 0 |
| 4 | 1 | 0 | `1 < 0`? Không | 0 |

Để ý: tới `i = 3` ta **đã gặp** min, nhưng cách 1 vẫn phải đọc nốt `i = 4` vì nó "không biết" mảng có cấu trúc gì — biết đâu phía sau còn số nhỏ hơn.

In [7]:
def trace_linear(nums):
    best = nums[0]
    print(f"i=0: nums[0]={nums[0]} -> khởi tạo best={best}")
    for i in range(1, len(nums)):
        x = nums[i]
        if x < best:
            print(f"i={i}: nums[{i}]={x} < best={best} -> cập nhật best={x}")
            best = x
        else:
            print(f"i={i}: nums[{i}]={x} >= best={best} -> giữ nguyên")
    print("Kết quả:", best, "| số phần tử đã đọc:", len(nums))
    return best


_ = trace_linear([2, 2, 2, 0, 1])

i=0: nums[0]=2 -> khởi tạo best=2
i=1: nums[1]=2 >= best=2 -> giữ nguyên
i=2: nums[2]=2 >= best=2 -> giữ nguyên
i=3: nums[3]=0 < best=2 -> cập nhật best=0
i=4: nums[4]=1 >= best=0 -> giữ nguyên
Kết quả: 0 | số phần tử đã đọc: 5


### Cách 1 vướng ở đâu?

- **Thời gian:** `O(n)` — luôn đọc đủ `n` phần tử. **Bộ nhớ:** `O(1)` (bản `nums[1:]` tạo bản sao `O(n)`; nếu muốn tuyệt đối `O(1)` thì duyệt bằng index).
- Với `n <= 5000`, `O(n)` thực ra **chạy rất nhanh** và được LeetCode chấp nhận. Nói thật để bạn khỏi lo: cách 1 *không* bị Time Limit Exceeded.
- Nhưng đề yêu cầu *giảm số bước càng nhiều càng tốt*, và người phỏng vấn hỏi bài này để xem bạn có **khai thác được cấu trúc "đã gần như sắp xếp"** hay không. Cách 1 bỏ phí hoàn toàn thông tin đó.

Câu hỏi dẫn đường: *thông tin "mảng là hai bậc thang" giúp ta bỏ qua được phần nào?*

## 6. Cách 2 — Tìm "điểm gãy" `nums[i] > nums[i+1]`

Từ mục 3: trong mảng xoay, các bước đi lên hoặc đi ngang **đều nằm trong một bậc thang**; chỉ có **đúng một chỗ** có thể "rơi xuống": chỗ nối từ cuối phần A sang đầu phần B.

> **Quan sát:** một mảng không giảm bị xoay có **nhiều nhất một** vị trí `i` với `nums[i] > nums[i+1]`. Nếu có, `nums[i+1]` chính là min. Nếu không có (mảng xoay `n` lần, hoặc mọi phần tử bằng nhau), min là `nums[0]`.

**Vì sao chỉ có nhiều nhất một chỗ gãy?** Bên trong phần A và bên trong phần B, mảng không giảm nên không có chỗ rơi. Chỉ còn đúng một cặp kề nhau "bắc cầu" giữa hai phần: (phần tử cuối của A, phần tử đầu của B).

**Ví dụ số:** `[2,2,2,0,1]`: cặp `(2,2)`, `(2,2)` không rơi; cặp `(2,0)` ở `i = 2` rơi → trả về `nums[3] = 0`, **dừng ngay**, không cần đọc `nums[4]`.

Với phần tử trùng, chú ý dấu: phải là `>` (rơi **thật sự**), không phải `>=`. Cặp `(2,2)` là bậc phẳng, không phải chỗ gãy.

In [8]:
def find_min_break_point(nums: List[int]) -> int:
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:      # chỗ "rơi xuống" duy nhất
            return nums[i + 1]
    return nums[0]                      # không có chỗ gãy: mảng đã sắp xếp


for nums in ([1, 3, 5], [2, 2, 2, 0, 1], [4, 5, 6, 7, 0, 1, 4], [1, 1, 1, 0, 1, 1], [3, 3, 3], [7]):
    print(nums, "->", find_min_break_point(nums))

[1, 3, 5] -> 1
[2, 2, 2, 0, 1] -> 0
[4, 5, 6, 7, 0, 1, 4] -> 0
[1, 1, 1, 0, 1, 1] -> 0
[3, 3, 3] -> 3
[7] -> 7


### Trace cách 2

Ta trace hai mảng: `[4,5,6,7,0,1,4]` (gãy ở giữa) và `[1,3,5]` (không gãy, phải đi hết rồi trả `nums[0]`).

| mảng | i | cặp so sánh | rơi? | hành động |
|---|---:|---|---|---|
| `[4,5,6,7,0,1,4]` | 0 | (4,5) | không | đi tiếp |
| | 1 | (5,6) | không | đi tiếp |
| | 2 | (6,7) | không | đi tiếp |
| | 3 | (7,0) | **có** | trả `nums[4] = 0` |
| `[1,3,5]` | 0 | (1,3) | không | đi tiếp |
| | 1 | (3,5) | không | hết vòng → trả `nums[0] = 1` |

In [9]:
def trace_break_point(nums):
    print("nums =", nums)
    for i in range(len(nums) - 1):
        a, b = nums[i], nums[i + 1]
        if a > b:
            print(f"  i={i}: ({a},{b}) rơi xuống -> min = nums[{i+1}] = {b}  (đã so {i+1} cặp)")
            return b
        print(f"  i={i}: ({a},{b}) không rơi -> đi tiếp")
    print(f"  không có chỗ gãy -> min = nums[0] = {nums[0]}  (đã so {len(nums)-1} cặp)")
    return nums[0]


_ = trace_break_point([4, 5, 6, 7, 0, 1, 4])
_ = trace_break_point([1, 3, 5])

nums = [4, 5, 6, 7, 0, 1, 4]
  i=0: (4,5) không rơi -> đi tiếp
  i=1: (5,6) không rơi -> đi tiếp
  i=2: (6,7) không rơi -> đi tiếp
  i=3: (7,0) rơi xuống -> min = nums[4] = 0  (đã so 4 cặp)
nums = [1, 3, 5]
  i=0: (1,3) không rơi -> đi tiếp
  i=1: (3,5) không rơi -> đi tiếp
  không có chỗ gãy -> min = nums[0] = 1  (đã so 2 cặp)


### Cách 2 vướng ở đâu?

- Tốt hơn cách 1 ở chỗ **dừng sớm** khi gặp chỗ gãy. Nhưng nếu chỗ gãy nằm gần **cuối** mảng (hoặc không có chỗ gãy), ta vẫn đi gần hết `n` phần tử. **Worst case vẫn `O(n)`**, bộ nhớ `O(1)`.
- Vấn đề gốc rễ: ta vẫn đi **từng bước một** từ trái sang phải. Muốn nhanh hơn hẳn, ta cần một cách **nhảy** tới giữa mảng, nhìn một phần tử, rồi **loại bỏ cả một nửa** mảng.

Đó chính là **binary search**. Trước khi áp vào bài, ta học nó từ đầu.

## 7. Nền tảng: Binary search (tìm kiếm nhị phân) từ căn bản

### 7.1. Trò chơi đoán số

Bạn bè nghĩ một số từ `1` đến `100`. Mỗi lần bạn đoán, họ chỉ trả lời "lớn hơn", "nhỏ hơn" hoặc "đúng".

- Cách ngây thơ: đoán `1, 2, 3, ...` → tệ nhất `100` lần.
- Cách thông minh: luôn đoán **số ở giữa** khoảng còn lại. Đoán `50`, nghe "lớn hơn" → khoảng còn `51..100`; đoán `75`... Mỗi lần **loại đi một nửa** khoảng.

Sau `k` lần, khoảng còn khoảng `100 / 2^k` số. Chỉ cần `2^k >= 100`, tức `k = 7` lần là chắc chắn tìm ra (`2^7 = 128`). Số lần đó là `log2(100) ≈ 6.6`, làm tròn lên. Ta gọi đó là độ phức tạp `O(log n)`.

Với `n = 5000` (giới hạn của đề): `log2(5000) ≈ 12.3` → khoảng **13 bước** thay vì 5000!

In [10]:
import math


def guess_game(secret, low=1, high=100):
    steps = 0
    while True:
        steps += 1
        guess = (low + high) // 2           # đoán số ở giữa
        if guess == secret:
            return steps
        if guess < secret:                  # "lớn hơn" -> bỏ nửa dưới
            low = guess + 1
        else:                               # "nhỏ hơn" -> bỏ nửa trên
            high = guess - 1


worst = max(guess_game(s) for s in range(1, 101))
print("Số lần đoán tệ nhất với 1..100:", worst)
print("log2(100)  =", round(math.log2(100), 2))
print("log2(5000) =", round(math.log2(5000), 2))

Số lần đoán tệ nhất với 1..100: 7
log2(100)  = 6.64
log2(5000) = 12.29


### 7.2. Binary search trên mảng đã sắp xếp: `lo`, `hi`, `mid`

Áp dụng trò chơi lên một mảng **đã sắp xếp** để tìm `target`:

- `lo`, `hi` là hai đầu (chỉ số) của **vùng còn nghi ngờ** (search space). Ban đầu `lo = 0`, `hi = n - 1`.
- `mid = (lo + hi) // 2` là chỉ số ở giữa. Toán tử `//` là **chia lấy phần nguyên** (`7 // 2 = 3`), nên `mid` luôn là số nguyên và khi vùng có số phần tử chẵn thì `mid` lệch về **bên trái**.
- So `nums[mid]` với `target`: nhỏ hơn → bỏ nửa trái (`lo = mid + 1`), lớn hơn → bỏ nửa phải (`hi = mid - 1`).

**Ví dụ số:** tìm `23` trong `[2,5,8,12,16,23,38,56,72,91]`:

| bước | lo | hi | mid | nums[mid] | quyết định |
|---:|---:|---:|---:|---:|---|
| 1 | 0 | 9 | 4 | 16 | `16 < 23` → `lo = 5` |
| 2 | 5 | 9 | 7 | 56 | `56 > 23` → `hi = 6` |
| 3 | 5 | 6 | 5 | 23 | tìm thấy ở index 5 |

Chỉ **3 lần nhìn** thay vì 6 (nếu đi từ trái).

In [11]:
def trace_binary_search(nums, target):
    lo, hi = 0, len(nums) - 1
    step = 0
    while lo <= hi:
        step += 1
        mid = (lo + hi) // 2
        print(f"bước {step}: lo={lo}, hi={hi}, mid={mid}, nums[mid]={nums[mid]}", end="  ")
        if nums[mid] == target:
            print("-> tìm thấy!")
            return mid
        if nums[mid] < target:
            print(f"-> {nums[mid]} < {target}, bỏ nửa trái: lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"-> {nums[mid]} > {target}, bỏ nửa phải: hi = {mid - 1}")
            hi = mid - 1
    print("-> không có")
    return -1


print("7 // 2 =", 7 // 2, "| (5 + 6) // 2 =", (5 + 6) // 2)
_ = trace_binary_search([2, 5, 8, 12, 16, 23, 38, 56, 72, 91], 23)

7 // 2 = 3 | (5 + 6) // 2 = 5
bước 1: lo=0, hi=9, mid=4, nums[mid]=16  -> 16 < 23, bỏ nửa trái: lo = 5
bước 2: lo=5, hi=9, mid=7, nums[mid]=56  -> 56 > 23, bỏ nửa phải: hi = 6
bước 3: lo=5, hi=6, mid=5, nums[mid]=23  -> tìm thấy!


### 7.3. Mẫu quan trọng nhất: "tìm vị trí `True` đầu tiên"

Binary search **không chỉ** dùng để tìm một giá trị. Sức mạnh thật sự của nó: nếu mỗi vị trí `i` có một câu hỏi Có/Không `cond(i)`, và câu trả lời có dạng

```text
i:        0      1      2      3     4     5     6
cond(i):  False  False  False  True  True  True  True
                               ^ cần tìm
```

(tất cả `False` đứng trước, tất cả `True` đứng sau — gọi là **monotone predicate**, "điều kiện đơn điệu"), thì binary search tìm được vị trí `True` đầu tiên trong `O(log n)`.

Mẫu code (khi biết chắc có ít nhất một `True`, và `hi` là chỉ số hợp lệ):

```python
lo, hi = 0, n - 1
while lo < hi:               # dừng khi vùng chỉ còn 1 vị trí
    mid = (lo + hi) // 2
    if cond(mid):            # mid là True -> đáp án ở mid hoặc bên trái
        hi = mid             #   (KHÔNG được bỏ mid: biết đâu mid là True đầu tiên)
    else:                    # mid là False -> đáp án chắc chắn ở bên phải mid
        lo = mid + 1
return lo
```

Ba chi tiết cần nhớ (sẽ gặp lại y nguyên trong bài):

1. `while lo < hi` (không có dấu `=`): vòng lặp dừng khi `lo == hi`, lúc đó vùng chỉ còn một ứng viên — chính là đáp án.
2. `hi = mid` chứ **không** phải `mid - 1`, vì `mid` có thể là đáp án.
3. `lo = mid + 1` vì `mid` chắc chắn **không** phải đáp án. Bước này cũng đảm bảo vùng luôn thu nhỏ (do `mid < hi` khi `lo < hi`), nên không lặp vô hạn.

Ví dụ: tìm vị trí đầu tiên có giá trị `>= 5` trong `[1,3,3,5,8,8,9]`.

Trong cell dưới có cú pháp `lambda i: a[i] >= 5`. Đây là cách viết **hàm ngắn một dòng không cần tên**, tương đương với `def cond(i): return a[i] >= 5`. Ta truyền hàm này vào `first_true` để nó hỏi "vị trí `mid` có thỏa điều kiện không?".

In [12]:
def first_true(n, cond, verbose=False):
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if verbose:
            print(f"lo={lo}, hi={hi}, mid={mid}, cond(mid)={cond(mid)}", end="  ")
        if cond(mid):
            hi = mid
            if verbose: print(f"-> hi = {hi}")
        else:
            lo = mid + 1
            if verbose: print(f"-> lo = {lo}")
    return lo


a = [1, 3, 3, 5, 8, 8, 9]
print("cond từng vị trí:", [x >= 5 for x in a])
idx = first_true(len(a), lambda i: a[i] >= 5, verbose=True)
print("Vị trí True đầu tiên:", idx, "| giá trị:", a[idx])

cond từng vị trí: [False, False, False, True, True, True, True]
lo=0, hi=6, mid=3, cond(mid)=True  -> hi = 3
lo=0, hi=3, mid=1, cond(mid)=False  -> lo = 2
lo=2, hi=3, mid=2, cond(mid)=False  -> lo = 3
Vị trí True đầu tiên: 3 | giá trị: 5


## 8. Cách 3 — Binary search cho bản KHÔNG trùng (LeetCode 153)

Tạm thời giả sử **mọi phần tử khác nhau** (đây chính là bài 153). Câu hỏi: có điều kiện Có/Không nào có dạng `False...False True...True` mà vị trí `True` đầu tiên là **min** không?

Có! Gọi `last = nums[-1]` (phần tử cuối mảng). Xét câu hỏi `cond(i) = nums[i] <= last`:

- Mọi phần tử của **phần A** đều **lớn hơn** `last` (vì `last` thuộc phần B, và A > B khi không trùng) → `False`.
- Mọi phần tử của **phần B** đều `<= last` (B tăng dần và `last` là phần tử cuối của B) → `True`.

```text
nums:          4      5      6      7      0     1     2
phần:          A      A      A      A      B     B     B
nums[i] <= 2?  False  False  False  False  True  True  True
                                           ^ True đầu tiên = min
```

Vậy tìm min = tìm `True` đầu tiên. Thay vì so với `nums[-1]` cố định, người ta thường so với `nums[hi]` (đầu phải của vùng đang xét) — kết quả giống hệt nhau cho bản không trùng, và cách viết này mở đường cho bản có trùng ở mục 10.

In [13]:
nums = [4, 5, 6, 7, 0, 1, 2]
last = nums[-1]
for i, x in enumerate(nums):
    print(f"i={i}: nums[i]={x}  nums[i] <= {last}? {x <= last}")

i=0: nums[i]=4  nums[i] <= 2? False
i=1: nums[i]=5  nums[i] <= 2? False
i=2: nums[i]=6  nums[i] <= 2? False
i=3: nums[i]=7  nums[i] <= 2? False
i=4: nums[i]=0  nums[i] <= 2? True
i=5: nums[i]=1  nums[i] <= 2? True
i=6: nums[i]=2  nums[i] <= 2? True


### Hai trường hợp khi so `nums[mid]` với `nums[hi]` (không trùng)

**Trường hợp `nums[mid] > nums[hi]`** — `mid` đang đứng ở phần A (cao), chỗ gãy nằm **bên phải** `mid`:

```text
        mid            hi
         v             v
 [ 4  5  6  7  0  1  2 ]          nums[mid]=6 > nums[hi]=2
               ^ min nằm trong (mid, hi]  ->  lo = mid + 1
```

`mid` lớn hơn một phần tử khác (`nums[hi]`) nên chắc chắn **không** phải min → bỏ luôn `mid`: `lo = mid + 1`.

**Trường hợp `nums[mid] < nums[hi]`** — đoạn từ `mid` tới `hi` đi lên liền mạch, **không** có chỗ gãy, nên các phần tử sau `mid` đều lớn hơn `nums[mid]`:

```text
  lo        mid     hi
  v          v      v
 [ 6  7  0  1  2  3  4 ]          nums[mid]=1 < nums[hi]=4
         ^ min nằm trong [lo, mid]  ->  hi = mid
```

`mid` có thể chính là min (ví dụ nếu `mid` đứng ngay sau chỗ gãy) → giữ lại `mid`: `hi = mid`.

Khi không trùng, trường hợp `nums[mid] == nums[hi]` chỉ xảy ra khi `mid == hi`, mà điều này không thể xảy ra trong vòng `while lo < hi` (vì `mid < hi`). Nên hai trường hợp trên là đủ.

In [14]:
def find_min_no_dup(nums: List[int]) -> int:
    """LeetCode 153: mảng xoay KHÔNG có phần tử trùng."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:     # mid ở phần A -> min ở bên phải mid
            lo = mid + 1
        else:                        # nums[mid] < nums[hi] -> min ở [lo, mid]
            hi = mid
    return nums[lo]


# Ví dụ của LeetCode 153
assert find_min_no_dup([3, 4, 5, 1, 2]) == 1
assert find_min_no_dup([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min_no_dup([11, 13, 15, 17]) == 11
assert find_min_no_dup([2, 1]) == 1
assert find_min_no_dup([5]) == 5
print("find_min_no_dup PASS các ví dụ của bài 153")

find_min_no_dup PASS các ví dụ của bài 153


### Trace cách 3 trên `[4,5,6,7,0,1,2]`

| bước | lo | hi | mid | nums[mid] | nums[hi] | so sánh | quyết định |
|---:|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 6 | 3 | 7 | 2 | `7 > 2` | `lo = 4` |
| 2 | 4 | 6 | 5 | 1 | 2 | `1 < 2` | `hi = 5` |
| 3 | 4 | 5 | 4 | 0 | 1 | `0 < 1` | `hi = 4` |
| — | 4 | 4 | | | | `lo == hi` | trả `nums[4] = 0` |

3 vòng lặp cho 7 phần tử. Với `n = 5000` chỉ khoảng 12–13 vòng.

In [15]:
def trace_find_min_no_dup(nums):
    lo, hi = 0, len(nums) - 1
    step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            action = f"{nums[mid]} > {nums[hi]} -> lo = {mid + 1}"
            new_lo, new_hi = mid + 1, hi
        else:
            sign = "<" if nums[mid] < nums[hi] else "=="
            action = f"{nums[mid]} {sign} {nums[hi]} -> (nhánh else) hi = {mid}"
            new_lo, new_hi = lo, mid
        print(f"bước {step}: lo={lo}, hi={hi}, mid={mid} | {action}")
        lo, hi = new_lo, new_hi
    print(f"dừng: lo = hi = {lo} -> min = {nums[lo]}")
    return nums[lo]


_ = trace_find_min_no_dup([4, 5, 6, 7, 0, 1, 2])

bước 1: lo=0, hi=6, mid=3 | 7 > 2 -> lo = 4
bước 2: lo=4, hi=6, mid=5 | 1 < 2 -> (nhánh else) hi = 5
bước 3: lo=4, hi=5, mid=4 | 0 < 1 -> (nhánh else) hi = 4
dừng: lo = hi = 4 -> min = 0


### Vì sao so với `nums[hi]` (bên phải) mà không so với `nums[lo]` (bên trái)?

Nhiều người thử viết: "nếu `nums[mid] >= nums[lo]` thì nửa trái đang tăng → min ở bên phải → `lo = mid + 1`". Nghe hợp lý, nhưng **sai** với mảng xoay `n` lần (đã sắp xếp), ví dụ `[1,2,3,4,5]`: nửa trái tăng nhưng min lại ở **đầu** nửa trái!

So với bên trái không phân biệt được hai tình huống "chưa xoay" và "gãy ở bên phải", vì cả hai đều có `nums[mid] >= nums[lo]`. So với bên phải thì luôn rõ ràng: `nums[mid] > nums[hi]` **chỉ** xảy ra khi có chỗ gãy giữa `mid` và `hi`.

In [16]:
def wrong_compare_left(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= nums[lo]:    # SAI: "nửa trái tăng -> min ở bên phải"
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]


for nums in ([1, 2, 3, 4, 5], [3, 4, 5, 1, 2], [4, 5, 6, 7, 0, 1, 2]):
    print(nums, "-> so bên trái:", wrong_compare_left(nums), "| đúng:", min(nums))

[1, 2, 3, 4, 5] -> so bên trái: 5 | đúng: 1
[3, 4, 5, 1, 2] -> so bên trái: 2 | đúng: 1
[4, 5, 6, 7, 0, 1, 2] -> so bên trái: 2 | đúng: 0


## 9. Khi có phần tử trùng: cách 3 hỏng ở đâu?

Bây giờ cho phép trùng. Xuất hiện một trường hợp mới mà cách 3 chưa nghĩ tới: **`nums[mid] == nums[hi]` trong khi `mid < hi`**.

Ví dụ "phá đám" quen thuộc: hai mảng dưới đây, ở bước đầu tiên (`lo = 0`, `hi = 5`, `mid = 2`) trông **giống hệt nhau** từ góc nhìn của thuật toán: `nums[mid] = 1`, `nums[hi] = 1`.

```text
              lo    mid       hi
 mảng X:    [ 1  1  1  0  1  1 ]     min ở index 3: BÊN PHẢI mid
 mảng Y:    [ 1  0  1  1  1  1 ]     min ở index 1: BÊN TRÁI mid
```

Cùng một quan sát (`nums[mid] == nums[hi]`) mà min có thể nằm **bên trái** hoặc **bên phải** `mid` → ta **không có quyền** bỏ nửa nào cả.

Cách 3 khi gặp `==` sẽ rơi vào nhánh `else` (`hi = mid`), tức là **bỏ nửa phải** — sai với mảng X. Xem cell dưới.

In [17]:
X = [1, 1, 1, 0, 1, 1]
Y = [1, 0, 1, 1, 1, 1]
for nums in (X, Y, [3, 3, 1, 3], [2, 2, 2, 0, 1]):
    got = find_min_no_dup(nums)
    print(f"{nums}: find_min_no_dup = {got}, đúng = {min(nums)}  ->", "OK" if got == min(nums) else "SAI!")

[1, 1, 1, 0, 1, 1]: find_min_no_dup = 1, đúng = 0  -> SAI!
[1, 0, 1, 1, 1, 1]: find_min_no_dup = 0, đúng = 0  -> OK
[3, 3, 1, 3]: find_min_no_dup = 3, đúng = 1  -> SAI!
[2, 2, 2, 0, 1]: find_min_no_dup = 0, đúng = 0  -> OK


Thú vị: `[2,2,2,0,1]` (Example 2) vẫn ra đúng — vì ở đó không lần nào rơi vào `nums[mid] == nums[hi]`. Lỗi chỉ lộ ra khi hai đầu có **giá trị bằng nhau** như `[1,1,1,0,1,1]` hay `[3,3,1,3]`. Đây là lý do test ví dụ của đề thôi là **chưa đủ**!

Ta trace cách 3 trên `[3,3,1,3]` để thấy chính xác bước nào đi sai:

In [18]:
_ = trace_find_min_no_dup([3, 3, 1, 3])
print("(Thật ra min = 1 ở index 2. Ngay bước 1, nums[mid] == nums[hi] rơi vào nhánh else và bỏ mất nửa phải chứa số 1.)")

bước 1: lo=0, hi=3, mid=1 | 3 == 3 -> (nhánh else) hi = 1
bước 2: lo=0, hi=1, mid=0 | 3 == 3 -> (nhánh else) hi = 0
dừng: lo = hi = 0 -> min = 3
(Thật ra min = 1 ở index 2. Ngay bước 1, nums[mid] == nums[hi] rơi vào nhánh else và bỏ mất nửa phải chứa số 1.)


## 10. Cách 4 — So `mid` với `hi`, bằng nhau thì `hi -= 1` (cách chính)

Khi `nums[mid] == nums[hi]`, ta không biết bỏ nửa nào. Nhưng ta vẫn biết **một điều nhỏ**: phần tử ở vị trí `hi` có thể **bỏ đi một cách an toàn**. Thuật toán:

| so sánh | ý nghĩa | hành động |
|---|---|---|
| `nums[mid] > nums[hi]` | `mid` ở phần A, chỗ gãy nằm trong `(mid, hi]` | `lo = mid + 1` |
| `nums[mid] < nums[hi]` | đoạn `[mid, hi]` không có chỗ gãy, min ở `[lo, mid]` | `hi = mid` |
| `nums[mid] == nums[hi]` | không biết bên nào | `hi -= 1` (chỉ bỏ **một** phần tử) |

### Vì sao `hi -= 1` an toàn?

Ta chỉ cần trả về **giá trị** min, không cần chỉ số. Xét phần tử `nums[hi]` đang bị bỏ:

- **Nếu `nums[hi]` không phải giá trị nhỏ nhất** → bỏ nó đi chẳng mất gì.
- **Nếu `nums[hi]` là giá trị nhỏ nhất** → vì `nums[mid] == nums[hi]`, nên `nums[mid]` **cũng** bằng giá trị nhỏ nhất. Và `mid < hi` (do `lo < hi` và `mid` lệch trái), nên `mid` vẫn nằm trong vùng mới `[lo, hi - 1]`. Giá trị min vẫn còn một "bản sao" trong vùng!

```text
 trước:  [ lo ... mid ... hi ]       nums[mid] == nums[hi] == x
 sau:    [ lo ... mid ... ] hi-1     x vẫn còn ở vị trí mid
```

Vậy sau mỗi bước, **giá trị nhỏ nhất của vùng `[lo, hi]` vẫn bằng giá trị nhỏ nhất của cả mảng**. Đó là invariant (bất biến) của thuật toán.

Còn hai nhánh `>` và `<`: lý luận giống hệt cách 3 (nhánh `>` bỏ những phần tử chắc chắn không phải min, nhánh `<` bỏ những phần tử `>= nums[mid]`). Lập luận đầy đủ cho cả ba nhánh nằm ở mục 14.

In [19]:
def find_min_binary(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:       # chỗ gãy ở bên phải mid
            lo = mid + 1
        elif nums[mid] < nums[hi]:     # [mid..hi] đi lên liền mạch, min ở [lo..mid]
            hi = mid
        else:                          # bằng nhau: không biết bên nào, bỏ đúng 1 phần tử
            hi -= 1
    return nums[lo]


assert find_min_binary([1, 3, 5]) == 1          # Example 1
assert find_min_binary([2, 2, 2, 0, 1]) == 0    # Example 2
assert find_min_binary([1, 1, 1, 0, 1, 1]) == 0
assert find_min_binary([1, 0, 1, 1, 1, 1]) == 0
assert find_min_binary([3, 3, 1, 3]) == 1
assert find_min_binary([10, 1, 10, 10, 10]) == 1
assert find_min_binary([4, 5, 6, 7, 0, 1, 4]) == 0
assert find_min_binary([2, 2, 2, 2]) == 2
assert find_min_binary([7]) == 7
print("find_min_binary PASS!")

find_min_binary PASS!


### Trace cách 4 trên `[4,5,6,7,1,3,3,3]` — gặp đủ cả ba nhánh

Mảng này là `[1,3,3,3,4,5,6,7]` xoay 4 lần. Phần A = `[4,5,6,7]`, phần B = `[1,3,3,3]`, min = `1` ở index 4.

| bước | lo | hi | mid | nums[mid] | nums[hi] | nhánh | quyết định | vùng mới |
|---:|---:|---:|---:|---:|---:|---|---|---|
| 1 | 0 | 7 | 3 | 7 | 3 | `>` | `lo = 4` | [4..7] |
| 2 | 4 | 7 | 5 | 3 | 3 | `==` | `hi = 6` | [4..6] |
| 3 | 4 | 6 | 5 | 3 | 3 | `==` | `hi = 5` | [4..5] |
| 4 | 4 | 5 | 4 | 1 | 3 | `<` | `hi = 4` | [4..4] |
| — | 4 | 4 | | | | dừng | trả `nums[4] = 1` | |

Chú ý bước 2 và 3: hai lần liền rơi vào nhánh `==`, mỗi lần chỉ thu nhỏ vùng **một** phần tử (thay vì một nửa). Nhưng nhờ vậy, `hi` "trượt" xuống tới chỗ `nums[hi]` khác `nums[mid]`, và ở bước 4 thuật toán lại có thông tin để quyết định.

In [20]:
def trace_find_min_binary(nums, show_window=True):
    lo, hi = 0, len(nums) - 1
    step = 0
    print("nums =", nums)
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        a, b = nums[mid], nums[hi]
        if a > b:
            branch, new_lo, new_hi = f"{a} > {b}  -> lo = {mid + 1}", mid + 1, hi
        elif a < b:
            branch, new_lo, new_hi = f"{a} < {b}  -> hi = {mid}", lo, mid
        else:
            branch, new_lo, new_hi = f"{a} == {b} -> hi = {hi - 1}", lo, hi - 1
        window = f" | vùng mới {nums[new_lo:new_hi + 1]}" if show_window else ""
        print(f"bước {step}: lo={lo}, hi={hi}, mid={mid} | {branch}{window}")
        lo, hi = new_lo, new_hi
    print(f"dừng sau {step} bước: lo = hi = {lo} -> min = {nums[lo]}")
    return nums[lo]


_ = trace_find_min_binary([4, 5, 6, 7, 1, 3, 3, 3])

nums = [4, 5, 6, 7, 1, 3, 3, 3]
bước 1: lo=0, hi=7, mid=3 | 7 > 3  -> lo = 4 | vùng mới [1, 3, 3, 3]
bước 2: lo=4, hi=7, mid=5 | 3 == 3 -> hi = 6 | vùng mới [1, 3, 3]
bước 3: lo=4, hi=6, mid=5 | 3 == 3 -> hi = 5 | vùng mới [1, 3]
bước 4: lo=4, hi=5, mid=4 | 1 < 3  -> hi = 4 | vùng mới [1]
dừng sau 4 bước: lo = hi = 4 -> min = 1


Bây giờ trace lại hai mảng "phá đám" `X = [1,1,1,0,1,1]` và `Y = [1,0,1,1,1,1]` — chúng giống nhau ở bước 1, nhưng nhờ nhánh `==` chỉ bỏ **một** phần tử, thuật toán không bao giờ "đánh cược" sai nửa:

In [21]:
_ = trace_find_min_binary([1, 1, 1, 0, 1, 1])
print()
_ = trace_find_min_binary([1, 0, 1, 1, 1, 1])

nums = [1, 1, 1, 0, 1, 1]
bước 1: lo=0, hi=5, mid=2 | 1 == 1 -> hi = 4 | vùng mới [1, 1, 1, 0, 1]
bước 2: lo=0, hi=4, mid=2 | 1 == 1 -> hi = 3 | vùng mới [1, 1, 1, 0]
bước 3: lo=0, hi=3, mid=1 | 1 > 0  -> lo = 2 | vùng mới [1, 0]
bước 4: lo=2, hi=3, mid=2 | 1 > 0  -> lo = 3 | vùng mới [0]
dừng sau 4 bước: lo = hi = 3 -> min = 0

nums = [1, 0, 1, 1, 1, 1]
bước 1: lo=0, hi=5, mid=2 | 1 == 1 -> hi = 4 | vùng mới [1, 0, 1, 1, 1]
bước 2: lo=0, hi=4, mid=2 | 1 == 1 -> hi = 3 | vùng mới [1, 0, 1, 1]
bước 3: lo=0, hi=3, mid=1 | 0 < 1  -> hi = 1 | vùng mới [1, 0]
bước 4: lo=0, hi=1, mid=0 | 1 > 0  -> lo = 1 | vùng mới [0]
dừng sau 4 bước: lo = hi = 1 -> min = 0


### Nhìn bằng hình: vùng `[lo, hi]` thu nhỏ dần

Cell dưới vẽ lại mảng `[4,5,6,7,1,3,3,3]` ở từng bước, đánh dấu `L` (lo), `M` (mid), `H` (hi). Nếu hai con trỏ trùng chỗ thì in ký tự `*`; dấu `.` là phần tử còn nằm trong vùng, ô trống là phần tử đã bị loại. Hãy để ý ở các bước `==`, `H` chỉ lùi đúng một ô.

In [22]:
def marks_of(lo, mid, hi):
    m = {}
    for idx, ch in ((lo, "L"), (mid, "M"), (hi, "H")):
        m[idx] = "*" if idx in m else ch
    return m


nums = [4, 5, 6, 7, 1, 3, 3, 3]
lo, hi = 0, len(nums) - 1
step = 0
while lo < hi:
    step += 1
    mid = (lo + hi) // 2
    m = marks_of(lo, mid, hi)
    print(f"bước {step}: lo={lo}, mid={mid}, hi={hi}")
    print("   nums: " + " ".join(str(v) for v in nums))
    print("         " + " ".join(m.get(i, "." if lo <= i <= hi else " ") for i in range(len(nums))))
    if nums[mid] > nums[hi]:
        lo = mid + 1
    elif nums[mid] < nums[hi]:
        hi = mid
    else:
        hi -= 1
print(f"kết thúc: lo=hi={lo}, min={nums[lo]}")

bước 1: lo=0, mid=3, hi=7
   nums: 4 5 6 7 1 3 3 3
         L . . M . . . H
bước 2: lo=4, mid=5, hi=7
   nums: 4 5 6 7 1 3 3 3
                 L M . H
bước 3: lo=4, mid=5, hi=6
   nums: 4 5 6 7 1 3 3 3
                 L M H  
bước 4: lo=4, mid=4, hi=5
   nums: 4 5 6 7 1 3 3 3
                 * H    
kết thúc: lo=hi=4, min=1


## 11. Vì sao worst case là `O(n)`? — trả lời câu *Follow up*

### 11.1. Đếm bước trên mảng nhiều phần tử trùng

Mỗi lần rơi vào nhánh `==`, vùng chỉ nhỏ đi **1** thay vì một nửa. Nếu **lần nào cũng** rơi vào `==` thì sao? Ví dụ mảng toàn số bằng nhau `[1,1,1,1,1,1]`: mọi so sánh đều `1 == 1`, `hi` lùi từng bước một từ `n - 1` về `0` → **`n - 1` bước**. Hay `[0,1,1,1,1,1]` (đã sắp xếp, min ở đầu): `hi` cũng phải lùi gần như từng bước một.

Cell dưới đếm số vòng lặp của `find_min_binary` trên ba loại mảng, với `n` tăng dần tới 5000:

- **phân biệt**: các số khác nhau, xoay `n // 3` lần,
- **một số 0**: toàn số `1`, chỉ có một số `0` ở đầu (`[0,1,1,...,1]`),
- **toàn bằng nhau**: `[1,1,...,1]`.

In [23]:
def count_steps(nums):
    lo, hi = 0, len(nums) - 1
    steps = 0
    while lo < hi:
        steps += 1
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        elif nums[mid] < nums[hi]:
            hi = mid
        else:
            hi -= 1
    return steps


print(f"{'n':>5} | {'log2(n)':>7} | {'phân biệt':>9} | {'một số 0':>8} | {'toàn bằng nhau':>14}")
for n in (10, 100, 1000, 5000):
    distinct = rotate(list(range(n)), n // 3)
    one_zero = [0] + [1] * (n - 1)
    all_same = [1] * n
    print(f"{n:>5} | {math.log2(n):>7.1f} | {count_steps(distinct):>9} | "
          f"{count_steps(one_zero):>8} | {count_steps(all_same):>14}")

    n | log2(n) | phân biệt | một số 0 | toàn bằng nhau
   10 |     3.3 |         3 |        9 |              9
  100 |     6.6 |         7 |       99 |             99
 1000 |    10.0 |        10 |      999 |            999
 5000 |    12.3 |        13 |     4999 |           4999


Kết quả nói rõ:

- Mảng **phân biệt**: số bước ≈ `log2(n)` → `O(log n)` như bài 153.
- Mảng có **rất nhiều phần tử trùng**: số bước ≈ `n - 1` → **`O(n)`**. Lúc này cách 4 không hơn gì cách 1 về mặt tiệm cận (nhưng cũng không tệ hơn).

### 11.2. Có thuật toán nào làm tốt hơn `O(n)` trong trường hợp xấu nhất không? — Không!

Lập luận "kẻ đối thủ" (adversary) đơn giản: xét các mảng dài `n` gồm toàn số `1`, chỉ có **một** số `0` đặt ở vị trí `p` bất kỳ:

```text
p = 0:  [0,1,1,1,1,1]
p = 1:  [1,0,1,1,1,1]
p = 2:  [1,1,0,1,1,1]
...
```

**Mọi** mảng như vậy đều hợp lệ (đều là `[0,1,1,...,1]` bị xoay), và còn có cả mảng `[1,1,...,1]` không có số `0` nào. Nếu một thuật toán **không đọc** vị trí `p`, nó không thể phân biệt "mảng có số 0 ở `p`" (đáp án `0`) với "mảng toàn số 1" (đáp án `1`). Vì `p` có thể là **bất kỳ** vị trí nào, trong trường hợp xấu nhất thuật toán phải đọc gần như **mọi** vị trí → `Ω(n)` (ký hiệu `Ω` nghĩa là "ít nhất cỡ").

Cell dưới kiểm tra rằng tất cả các mảng đó đều hợp lệ và cách 4 đều trả lời đúng.

In [24]:
n = 6
base_sorted = [0] + [1] * (n - 1)
valid_rotations = [rotate(base_sorted, k) for k in range(1, n + 1)]
for p in range(n):
    arr = [1] * n
    arr[p] = 0
    print(f"p={p}: {arr}  hợp lệ? {arr in valid_rotations}  | find_min_binary = {find_min_binary(arr)}"
          f"  ({count_steps(arr)} bước)")
print("Mảng toàn 1:", [1] * n, "-> find_min_binary =", find_min_binary([1] * n))

p=0: [0, 1, 1, 1, 1, 1]  hợp lệ? True  | find_min_binary = 0  (5 bước)
p=1: [1, 0, 1, 1, 1, 1]  hợp lệ? True  | find_min_binary = 0  (4 bước)
p=2: [1, 1, 0, 1, 1, 1]  hợp lệ? True  | find_min_binary = 0  (2 bước)
p=3: [1, 1, 1, 0, 1, 1]  hợp lệ? True  | find_min_binary = 0  (4 bước)
p=4: [1, 1, 1, 1, 0, 1]  hợp lệ? True  | find_min_binary = 0  (3 bước)
p=5: [1, 1, 1, 1, 1, 0]  hợp lệ? True  | find_min_binary = 0  (2 bước)
Mảng toàn 1: [1, 1, 1, 1, 1, 1] -> find_min_binary = 1


### 11.3. Trả lời *Follow up* bằng lời

> *Would this affect the runtime complexity? How and why?*

**Có.** Với bài 153 (không trùng), binary search luôn bỏ được một nửa mỗi bước → `O(log n)`. Với bài 154, khi `nums[mid] == nums[hi]` ta không biết min nằm nửa nào (ví dụ `[1,1,1,0,1,1]` và `[1,0,1,1,1,1]` trông giống nhau ở `mid` và `hi`), nên chỉ được bỏ **một** phần tử (`hi -= 1`). Trường hợp xấu nhất (rất nhiều phần tử trùng, như `[1,1,...,1]` hoặc `[1,...,1,0,1,...,1]`) cần `O(n)` bước. Và đây là **giới hạn của chính bài toán**, không phải do thuật toán kém: lập luận adversary ở trên cho thấy mọi thuật toán đều phải đọc `Ω(n)` phần tử trong trường hợp xấu nhất. Khi ít phần tử trùng, cách 4 vẫn chạy gần `O(log n)`.

## 12. Cách 5 (tùy chọn) — Chia để trị bằng đệ quy

### 12.1. Nền tảng: đệ quy (recursion)

**Đệ quy** là khi một hàm **tự gọi lại chính nó** với một bài toán nhỏ hơn. Mỗi hàm đệ quy cần hai phần:

1. **Trường hợp cơ sở** (base case): bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia bài toán thành bài nhỏ hơn, gọi lại hàm, rồi ghép kết quả.

Ví dụ mini: tìm max của đoạn `a[lo..hi]` bằng cách **chia đôi**: max của cả đoạn = max(max nửa trái, max nửa phải). Base case: đoạn chỉ có 1 phần tử (`lo == hi`).

In [25]:
def max_rec(a, lo, hi, depth=0):
    pad = "    " * depth                      # thụt lề theo độ sâu để thấy "cây gọi hàm"
    if lo == hi:                              # base case: 1 phần tử
        print(f"{pad}max_rec({lo},{hi}) = {a[lo]}   (base case)")
        return a[lo]
    mid = (lo + hi) // 2
    print(f"{pad}max_rec({lo},{hi}): chia thành [{lo}..{mid}] và [{mid + 1}..{hi}]")
    left = max_rec(a, lo, mid, depth + 1)
    right = max_rec(a, mid + 1, hi, depth + 1)
    print(f"{pad}max_rec({lo},{hi}) = max({left}, {right}) = {max(left, right)}")
    return max(left, right)


_ = max_rec([3, 8, 2, 5], 0, 3)

max_rec(0,3): chia thành [0..1] và [2..3]
    max_rec(0,1): chia thành [0..0] và [1..1]
        max_rec(0,0) = 3   (base case)
        max_rec(1,1) = 8   (base case)
    max_rec(0,1) = max(3, 8) = 8
    max_rec(2,3): chia thành [2..2] và [3..3]
        max_rec(2,2) = 2   (base case)
        max_rec(3,3) = 5   (base case)
    max_rec(2,3) = max(2, 5) = 5
max_rec(0,3) = max(8, 5) = 8


### 12.2. Áp vào bài: chia đôi + "cắt tỉa" đoạn đã sắp xếp

Nếu chỉ chia đôi rồi lấy `min` hai nửa thì phải thăm **mọi** phần tử → `O(n)`, chẳng hơn cách 1. Mấu chốt là một **phép cắt tỉa** (pruning):

> Nếu `nums[lo] < nums[hi]` thì đoạn `[lo..hi]` **không chứa chỗ gãy**, tức là đã sắp xếp → min của đoạn là `nums[lo]`, trả về ngay.

**Vì sao?** Nếu đoạn chứa chỗ gãy, thì `nums[lo]` thuộc phần A và `nums[hi]` thuộc phần B, mà mọi phần tử A `>=` mọi phần tử B (quan sát 2 ở mục 3) → `nums[lo] >= nums[hi]`. Vậy `nums[lo] < nums[hi]` chắc chắn nghĩa là **không** có chỗ gãy.

Khi không trùng: mỗi lần chia đôi, **nhiều nhất một** nửa chứa chỗ gãy, nửa kia được cắt tỉa ngay → khoảng `2·log2(n)` lần gọi hàm → `O(log n)`. Khi có nhiều phần tử trùng (`nums[lo] == nums[hi]`), phép cắt tỉa không áp dụng được và ta phải chia tiếp cả hai nửa → worst case `O(n)`. Đúng như kết luận ở mục 11!

In [26]:
def find_min_divide(nums: List[int]) -> int:
    def solve(lo, hi):
        if lo == hi or nums[lo] < nums[hi]:   # 1 phần tử, hoặc đoạn đã sắp xếp
            return nums[lo]
        mid = (lo + hi) // 2
        return min(solve(lo, mid), solve(mid + 1, hi))

    return solve(0, len(nums) - 1)


for nums in ([1, 3, 5], [2, 2, 2, 0, 1], [3, 3, 1, 3], [1, 1, 1, 0, 1, 1], [4, 5, 6, 7, 1, 3, 3, 3], [7]):
    print(nums, "->", find_min_divide(nums))

[1, 3, 5] -> 1
[2, 2, 2, 0, 1] -> 0
[3, 3, 1, 3] -> 1
[1, 1, 1, 0, 1, 1] -> 0
[4, 5, 6, 7, 1, 3, 3, 3] -> 1
[7] -> 7


### Trace cách 5: cây gọi hàm trên `[2,2,2,0,1]` và `[4,5,6,7,0,1,2]`

Với `[2,2,2,0,1]`:

```text
solve(0,4)  nums[0]=2 >= nums[4]=1  -> chia
├── solve(0,2)  nums[0]=2 >= nums[2]=2 -> chia
│   ├── solve(0,1)  2 >= 2 -> chia
│   │   ├── solve(0,0) = 2   (1 phần tử)
│   │   └── solve(1,1) = 2
│   └── solve(2,2) = 2
└── solve(3,4)  nums[3]=0 < nums[4]=1 -> đã sắp xếp, trả 0 (cắt tỉa)
kết quả = min(2, 0) = 0
```

Nhánh trái `[2,2,2]` toàn số bằng nhau nên **không cắt tỉa được** — phải đi tới tận lá. Nhánh phải `[0,1]` được cắt tỉa ngay.

In [27]:
def trace_divide(nums):
    calls = [0]

    def solve(lo, hi, depth):
        calls[0] += 1
        pad = "    " * depth
        if lo == hi:
            print(f"{pad}solve({lo},{hi}) = {nums[lo]}   (1 phần tử)")
            return nums[lo]
        if nums[lo] < nums[hi]:
            print(f"{pad}solve({lo},{hi}): {nums[lo]} < {nums[hi]} -> đã sắp xếp, trả {nums[lo]} (cắt tỉa)")
            return nums[lo]
        mid = (lo + hi) // 2
        print(f"{pad}solve({lo},{hi}): {nums[lo]} >= {nums[hi]} -> chia [{lo}..{mid}] và [{mid + 1}..{hi}]")
        res = min(solve(lo, mid, depth + 1), solve(mid + 1, hi, depth + 1))
        print(f"{pad}=> solve({lo},{hi}) = {res}")
        return res

    ans = solve(0, len(nums) - 1, 0)
    print(f"Kết quả: {ans} | số lần gọi solve: {calls[0]}")
    return ans


_ = trace_divide([2, 2, 2, 0, 1])
print()
_ = trace_divide([4, 5, 6, 7, 0, 1, 2])

solve(0,4): 2 >= 1 -> chia [0..2] và [3..4]
    solve(0,2): 2 >= 2 -> chia [0..1] và [2..2]
        solve(0,1): 2 >= 2 -> chia [0..0] và [1..1]
            solve(0,0) = 2   (1 phần tử)
            solve(1,1) = 2   (1 phần tử)
        => solve(0,1) = 2
        solve(2,2) = 2   (1 phần tử)
    => solve(0,2) = 2
    solve(3,4): 0 < 1 -> đã sắp xếp, trả 0 (cắt tỉa)
=> solve(0,4) = 0
Kết quả: 0 | số lần gọi solve: 7

solve(0,6): 4 >= 2 -> chia [0..3] và [4..6]
    solve(0,3): 4 < 7 -> đã sắp xếp, trả 4 (cắt tỉa)
    solve(4,6): 0 < 2 -> đã sắp xếp, trả 0 (cắt tỉa)
=> solve(0,6) = 0
Kết quả: 0 | số lần gọi solve: 3


### Độ phức tạp cách 5

- **Thời gian:** `O(log n)` khi không trùng (hoặc ít trùng); `O(n)` trong trường hợp xấu nhất (nhiều trùng, ví dụ `[1,1,...,1]` phải đi tới mọi lá: khoảng `2n - 1` lần gọi).
- **Bộ nhớ:** `O(log n)` cho **call stack** (ngăn xếp lời gọi hàm: mỗi lần gọi đệ quy chưa xong chiếm một ô nhớ), vì độ sâu cây chia đôi là `log2(n)`.

So với cách 4: cùng độ phức tạp tiệm cận, nhưng cách 4 dùng `O(1)` bộ nhớ và ít lời gọi hàm hơn → cách 4 là lựa chọn chính để nộp. Cách 5 đáng học vì nó cho thấy **cùng một quan sát** ("đoạn có `nums[lo] < nums[hi]` thì đã sắp xếp") có thể dùng theo kiểu đệ quy.

In [28]:
def count_calls_divide(nums):
    calls = 0

    def solve(lo, hi):
        nonlocal calls                      # cho phép hàm con sửa biến calls của hàm ngoài
        calls += 1
        if lo == hi or nums[lo] < nums[hi]:
            return nums[lo]
        mid = (lo + hi) // 2
        return min(solve(lo, mid), solve(mid + 1, hi))

    solve(0, len(nums) - 1)
    return calls


for n in (10, 100, 1000, 5000):
    print(f"n={n:>4}: mảng phân biệt -> {count_calls_divide(rotate(list(range(n)), n // 3)):>3} lần gọi | "
          f"toàn bằng nhau -> {count_calls_divide([1] * n):>5} lần gọi")

n=  10: mảng phân biệt ->   5 lần gọi | toàn bằng nhau ->    19 lần gọi
n= 100: mảng phân biệt ->  15 lần gọi | toàn bằng nhau ->   199 lần gọi
n=1000: mảng phân biệt ->  17 lần gọi | toàn bằng nhau ->  1999 lần gọi
n=5000: mảng phân biệt ->  21 lần gọi | toàn bằng nhau ->  9999 lần gọi


## 13. Code nộp LeetCode

Chuyển cách 4 vào đúng khuôn `class Solution`. Lưu ý:

- Tên method phải là `findMin`, tham số `self, nums: List[int]`, trả về `int`.
- Trên LeetCode, `List` đã được import sẵn; khi chạy trong notebook ta đã `from typing import List` ở đầu.
- Không in gì ra màn hình trong code nộp.

In [29]:
class Solution:
    def findMin(self, nums: List[int]) -> int:
        lo, hi = 0, len(nums) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] > nums[hi]:      # chỗ gãy nằm bên phải mid
                lo = mid + 1
            elif nums[mid] < nums[hi]:    # [mid..hi] đi lên liền mạch -> min ở [lo..mid]
                hi = mid
            else:                         # nums[mid] == nums[hi]: bỏ hi, giá trị vẫn còn ở mid
                hi -= 1
        return nums[lo]


sol = Solution()
# Ví dụ của đề
assert sol.findMin([1, 3, 5]) == 1
assert sol.findMin([2, 2, 2, 0, 1]) == 0
# Edge case
assert sol.findMin([5]) == 5                          # n = 1
assert sol.findMin([2, 1]) == 1                       # n = 2, có gãy
assert sol.findMin([1, 2]) == 1                       # n = 2, xoay n lần
assert sol.findMin([1, 1]) == 1                       # toàn bằng nhau
assert sol.findMin([1, 1, 1, 0, 1, 1]) == 0           # hai đầu bằng nhau, min bên phải mid
assert sol.findMin([1, 0, 1, 1, 1, 1]) == 0           # hai đầu bằng nhau, min bên trái mid
assert sol.findMin([3, 3, 1, 3]) == 1
assert sol.findMin([10, 1, 10, 10, 10]) == 1
assert sol.findMin([-5000, -5000, 5000]) == -5000     # giá trị biên của constraints
assert sol.findMin([5000, -5000, -5000]) == -5000
print("class Solution PASS toàn bộ test!")

class Solution PASS toàn bộ test!


### Kiểm thử đối chiếu chéo tất cả các cách

Ta sinh ngẫu nhiên mảng hợp lệ đúng theo đề: lấy một list ngẫu nhiên, **sắp xếp**, rồi **xoay `k` lần** với `k` ngẫu nhiên trong `1..n`. Giá trị chọn trong khoảng hẹp (`0..4`) để có **thật nhiều phần tử trùng** — đó là chỗ dễ sai nhất. `random.seed(154)` cố định để lần chạy nào cũng ra cùng bộ test.

Ngoài ra ta kiểm tra: `find_min_no_dup` (cách 3) đúng với mọi mảng **không trùng**, nhưng **có** mảng trùng làm nó sai.

In [30]:
import random


def random_rotated(n, low, high):
    base = sorted(random.randint(low, high) for _ in range(n))
    k = random.randint(1, n)                 # xoay từ 1 tới n lần, đúng như đề
    return rotate(base, k)


methods = [find_min_builtin, find_min_linear, find_min_break_point,
           find_min_binary, find_min_divide, Solution().findMin]

random.seed(154)
for _ in range(3000):
    nums = random_rotated(random.randint(1, 12), 0, 4)
    expected = min(nums)
    for f in methods:
        assert f(nums) == expected, (f.__name__, nums)
print("5 cách + class Solution khớp nhau trên 3000 mảng ngẫu nhiên có trùng")

no_dup_fail = 0
for _ in range(3000):
    n = random.randint(1, 12)
    distinct = rotate(sorted(random.sample(range(-50, 50), n)), random.randint(1, n))
    assert find_min_no_dup(distinct) == min(distinct)
    dup = random_rotated(n, 0, 4)
    no_dup_fail += find_min_no_dup(dup) != min(dup)
print("find_min_no_dup đúng trên 3000 mảng KHÔNG trùng")
print("find_min_no_dup sai trên", no_dup_fail, "/ 3000 mảng CÓ trùng -> không dùng được cho bài 154")

5 cách + class Solution khớp nhau trên 3000 mảng ngẫu nhiên có trùng


find_min_no_dup đúng trên 3000 mảng KHÔNG trùng
find_min_no_dup sai trên 17 / 3000 mảng CÓ trùng -> không dùng được cho bài 154


In [31]:
# Kiểm tra ở kích thước lớn nhất của constraints: n = 5000, giá trị trong [-5000, 5000]
random.seed(2026)
for _ in range(20):
    nums = random_rotated(5000, -5000, 5000)
    assert Solution().findMin(nums) == min(nums)
    assert find_min_divide(nums) == min(nums)
big_dup = rotate([-5000] + [7] * 4999, 1234)
assert Solution().findMin(big_dup) == -5000
print("PASS với n = 5000 | số bước trên mảng gần như toàn trùng:", count_steps(big_dup))

PASS với n = 5000 | số bước trên mảng gần như toàn trùng: 2541


## 14. Vì sao cách 4 đúng? (invariant bằng lời)

**Invariant (bất biến):** ở đầu mỗi vòng lặp, *giá trị nhỏ nhất của đoạn `nums[lo..hi]` bằng giá trị nhỏ nhất của cả mảng*.

- **Ban đầu** `lo = 0`, `hi = n - 1`: đoạn là cả mảng → đúng.
- **Nhánh `nums[mid] > nums[hi]`** (`lo = mid + 1`): đi từ `nums[mid]` tới `nums[hi]` nhỏ hơn thì phải có một chỗ "rơi xuống" ở giữa, tức là chỗ gãy (duy nhất) nằm trong đoạn `mid..hi`. Phần tử ngay sau chỗ gãy là **đầu phần B**, chính là vị trí của giá trị nhỏ nhất toàn mảng (mục 3, quan sát 3), và nó nằm trong `[mid + 1..hi]`. Vậy vùng mới vẫn chứa giá trị min. Ví dụ `[4,5,6,7,1,3,3,3]`, `mid = 3`: `7 > 3`, chỗ gãy `7 → 1` nằm giữa index 3 và 4, min ở index 4 thuộc vùng mới `[4..7]`.
- **Nhánh `nums[mid] < nums[hi]`** (`hi = mid`): nếu đoạn `(mid..hi]` chứa chỗ gãy thì `nums[mid]` thuộc A và `nums[hi]` thuộc B, kéo theo `nums[mid] >= nums[hi]` — mâu thuẫn. Vậy `[mid..hi]` không có chỗ gãy, là đoạn không giảm, mọi phần tử sau `mid` đều `>= nums[mid]`. Bỏ chúng, giữ `mid` → min vẫn còn.
- **Nhánh `nums[mid] == nums[hi]`** (`hi -= 1`): như mục 10 — nếu `nums[hi]` là min thì bản sao `nums[mid]` (với `mid < hi`) vẫn ở trong vùng mới.

**Dừng:** mỗi vòng, độ dài vùng `hi - lo` giảm ít nhất 1 (vì `lo <= mid < hi`, nên `mid + 1 > lo`, `mid < hi`, `hi - 1 < hi`). Vòng lặp chắc chắn dừng khi `lo == hi`.

**Kết luận:** khi dừng, vùng chỉ còn `nums[lo]`, và theo invariant đó chính là giá trị nhỏ nhất của cả mảng.

**Lưu ý tinh tế:** invariant nói về **giá trị**, không nói về **chỉ số**. Nhánh `==` có thể bỏ đúng vị trí "chỗ gãy" thật sự, nên `lo` cuối cùng **không nhất thiết** là số lần xoay. Với bài này điều đó không sao (đề chỉ hỏi giá trị); xem lỗi 6 ở mục 16 và bài tập 6.

## 15. So sánh các cách

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time (tốt nhất / xấu nhất) | Space | Ghi chú |
|---|---|---|---|---|
| 1. Quét tuyến tính / `min()` | đọc hết, nhớ min | `O(n)` / `O(n)` | `O(1)` | đúng với mọi list, làm baseline |
| 2. Tìm điểm gãy | dừng ở `nums[i] > nums[i+1]` | `O(1)` / `O(n)` | `O(1)` | dừng sớm nếu gãy ở đầu |
| 3. Binary search bản 153 | so `mid` với `hi`, hai nhánh | `O(log n)` | `O(1)` | **SAI** khi có trùng (ví dụ `[3,3,1,3]`) |
| 4. Binary search + `hi -= 1` | thêm nhánh `==` | `O(log n)` / `O(n)` | `O(1)` | **cách chính**, nộp LeetCode |
| 5. Chia để trị đệ quy | chia đôi + cắt tỉa đoạn đã sắp xếp | `O(log n)` / `O(n)` | `O(log n)` stack | cùng ý tưởng, viết kiểu đệ quy |

Với cách 4, "tốt nhất `O(log n)`" ở đây nghĩa là khi mảng ít trùng; trường hợp may mắn nhất (vd `n` nhỏ) thì còn ít bước hơn nữa. Điểm mấu chốt: cách 4 **không bao giờ tệ hơn** cách 1 về tiệm cận, và tốt hơn rất nhiều khi ít trùng.

## 16. Lỗi người mới hay gặp

### Lỗi 1: Dùng nguyên code bài 153 (gộp `==` vào `else: hi = mid`)

Đã thấy ở mục 9: `[3,3,1,3]` → trả `3` thay vì `1`; `[1,1,1,0,1,1]` → trả `1` thay vì `0`. **Sửa:** tách riêng nhánh `==` và chỉ `hi -= 1`.

### Lỗi 2: So `nums[mid]` với `nums[lo]` thay vì `nums[hi]`

Đã thấy ở mục 8: sai ngay với mảng xoay `n` lần như `[1,2,3,4,5]`. **Sửa:** so với `nums[hi]`.

### Lỗi 3: Nhánh `<` viết `hi = mid - 1`

`mid` có thể chính là min (ví dụ `[3,1,2]`: `mid = 1`, `nums[mid] = 1 < 2`). Viết `hi = mid - 1` là vứt mất min. **Sửa:** `hi = mid`.

### Lỗi 4: Nhánh `>` viết `lo = mid` → lặp vô hạn

Với `[2,1]`: `lo = 0`, `hi = 1`, `mid = 0`, `2 > 1` → `lo = 0`... vùng không bao giờ thu nhỏ. **Sửa:** `lo = mid + 1` (`mid` lớn hơn `nums[hi]` nên chắc chắn không phải min). Tương tự, viết `while lo <= hi` kèm `hi = mid` cũng gây lặp vô hạn khi `lo == hi`.

### Lỗi 5: Nhánh `==` viết `lo += 1`

Nghe có vẻ "đối xứng" nhưng **sai**: lý do `hi -= 1` an toàn là `nums[hi]` có bản sao `nums[mid]` trong vùng. Còn `nums[lo]` thì **không** chắc bằng `nums[mid]`. Ví dụ `[1,3,3]`: `mid = 1`, `3 == 3` → `lo += 1` vứt mất số `1`.

### Lỗi 6: Tưởng `lo` cuối cùng là "số lần xoay" / vị trí chỗ gãy

Với `[1,1,1,2,1]` (là `[1,1,1,1,2]` xoay 1 lần, chỗ gãy thật ở index 4), cách 4 trả **giá trị** đúng (`1`) nhưng `lo` dừng ở index `0`. Nếu đề hỏi chỉ số (như bài tập 6), cần xử lý thêm.

Cell dưới chạy thử các phiên bản sai (có giới hạn số vòng để không treo máy).

In [32]:
def buggy_min(nums, variant, max_steps=50):
    lo, hi = 0, len(nums) - 1
    steps = 0
    while lo < hi:
        steps += 1
        if steps > max_steps:
            return "LẶP VÔ HẠN"
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid if variant == "lo=mid" else mid + 1
        elif nums[mid] < nums[hi]:
            hi = mid - 1 if variant == "hi=mid-1" else mid
        else:
            if variant == "153":
                hi = mid
            elif variant == "lo+=1":
                lo += 1
            else:
                hi -= 1
    return nums[lo]


cases = [("153", [3, 3, 1, 3]), ("153", [1, 1, 1, 0, 1, 1]), ("hi=mid-1", [3, 1, 2]),
         ("lo=mid", [2, 1]), ("lo+=1", [1, 3, 3]), ("đúng", [1, 3, 3])]
for variant, nums in cases:
    print(f"phiên bản {variant:>8} trên {nums}: trả {buggy_min(nums, variant)}, đúng là {min(nums)}")

print("Lỗi 6: find_min_binary([1,1,1,2,1]) =", find_min_binary([1, 1, 1, 2, 1]))
_ = trace_find_min_binary([1, 1, 1, 2, 1], show_window=False)
print("-> giá trị đúng, nhưng lo = 0 trong khi chỗ gãy thật (2 -> 1) ở index 4")

phiên bản      153 trên [3, 3, 1, 3]: trả 3, đúng là 1
phiên bản      153 trên [1, 1, 1, 0, 1, 1]: trả 1, đúng là 0
phiên bản hi=mid-1 trên [3, 1, 2]: trả 3, đúng là 1
phiên bản   lo=mid trên [2, 1]: trả LẶP VÔ HẠN, đúng là 1
phiên bản    lo+=1 trên [1, 3, 3]: trả 3, đúng là 1
phiên bản     đúng trên [1, 3, 3]: trả 1, đúng là 1
Lỗi 6: find_min_binary([1,1,1,2,1]) = 1
nums = [1, 1, 1, 2, 1]
bước 1: lo=0, hi=4, mid=2 | 1 == 1 -> hi = 3
bước 2: lo=0, hi=3, mid=1 | 1 < 2  -> hi = 1
bước 3: lo=0, hi=1, mid=0 | 1 == 1 -> hi = 0
dừng sau 3 bước: lo = hi = 0 -> min = 1
-> giá trị đúng, nhưng lo = 0 trong khi chỗ gãy thật (2 -> 1) ở index 4


## 17. Mẹo nhận dạng pattern

- Thấy **"sorted" + "rotated"** → nghĩ ngay tới **binary search** trên hình "hai bậc thang".
- Thấy đề bắt **"giảm số bước"**, **"O(log n)"**, hoặc mảng **gần như sắp xếp** → đừng quét tuyến tính, hãy tìm câu hỏi Có/Không đơn điệu để chia đôi.
- Khi so sánh để chia đôi, hỏi: *"so với đầu nào thì kết luận luôn rõ ràng?"* — với bài tìm min, so với **đầu phải** (`nums[hi]`).
- Thấy **"may contain duplicates"** trong bài binary search → chuẩn bị nhánh **"bằng nhau thì thu hẹp 1 phần tử"** (ở đây `hi -= 1`, ở bài 81 là `lo += 1; hi -= 1`), và chấp nhận worst case `O(n)`.
- Muốn biết bước thu hẹp có an toàn không: hỏi *"phần tử bị bỏ, nếu nó là đáp án, thì còn bản sao/thay thế nào trong vùng không?"*.
- Test luôn có các mảng **hai đầu bằng nhau** (`[1,1,1,0,1,1]`, `[3,3,1,3]`), mảng **chưa xoay** (`[1,2,3]`), **toàn bằng nhau**, và `n = 1, 2`.

## 18. Biến thể và bài liên quan

| Bài | Nội dung | Liên hệ |
|---|---|---|
| 153. Find Minimum in Rotated Sorted Array | bản không trùng | cách 3, `O(log n)` |
| 33. Search in Rotated Sorted Array | tìm `target` trong mảng xoay không trùng | binary search, xác định nửa nào đã sắp xếp |
| 81. Search in Rotated Sorted Array II | như 33 nhưng có trùng, trả `True/False` | cùng "căn bệnh" trùng, sửa bằng `lo += 1; hi -= 1` (bài tập 5) |
| 162. Find Peak Element | tìm đỉnh | binary search trên điều kiện so sánh `nums[mid]` với hàng xóm |
| 852. Peak Index in a Mountain Array | mảng "núi" | binary search trên hình dạng mảng |
| 35. Search Insert Position | vị trí chèn | mẫu "tìm `True` đầu tiên" ở mục 7.3 |

Một biến thể hay hỏi phỏng vấn: **"mảng đã bị xoay bao nhiêu lần?"** — với mảng không trùng, đáp án chính là chỉ số `lo` mà cách 3 tìm được. Với mảng có trùng, xem bài tập 6.

In [33]:
# Biến thể nhỏ: số lần xoay của mảng KHÔNG trùng = chỉ số của min (lo của cách 3)
def rotation_count_no_dup(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return lo      # = số lần xoay, tính theo modulo n (xoay n lần <=> 0 lần)


base = [10, 20, 30, 40, 50]
for k in range(1, 6):
    arr = rotate(base, k)
    print(f"xoay {k} lần: {arr} -> rotation_count_no_dup = {rotation_count_no_dup(arr)} (= {k} % 5 = {k % 5})")

xoay 1 lần: [50, 10, 20, 30, 40] -> rotation_count_no_dup = 1 (= 1 % 5 = 1)
xoay 2 lần: [40, 50, 10, 20, 30] -> rotation_count_no_dup = 2 (= 2 % 5 = 2)
xoay 3 lần: [30, 40, 50, 10, 20] -> rotation_count_no_dup = 3 (= 3 % 5 = 3)
xoay 4 lần: [20, 30, 40, 50, 10] -> rotation_count_no_dup = 4 (= 4 % 5 = 4)
xoay 5 lần: [10, 20, 30, 40, 50] -> rotation_count_no_dup = 0 (= 5 % 5 = 0)


## 19. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 20, đáp án ở mục 21 (cuối notebook).

### Bài 1 — Tính tay (không chạy code)

Lập bảng `lo, hi, mid, nums[mid], nums[hi], nhánh, quyết định` của **cách 4** (`find_min_binary`) cho:

- (a) `[2,2,0,0,1,2,2]`
- (b) `[3,3,1,3]`

Mỗi mảng dừng sau bao nhiêu bước? Mảng (a) có đi qua đủ cả ba nhánh `>`, `<`, `==` không?

### Bài 2 — Đoán output

Không chạy code, đoán:

- (a) `find_min_no_dup([1,1,1,0,1,1])` trả về gì? (Cách 3, bản dành cho bài 153.)
- (b) `count_steps([7,7,7,7,7,7,7,7])` (8 số 7) là bao nhiêu?
- (c) `count_steps([5,6,7,1,2,3,4])` là bao nhiêu?

### Bài 3 — Điền TODO: cách 4

Điền 3 nhánh của vòng lặp. Đừng nhìn lại mục 10 nếu có thể!

In [34]:
def exercise_find_min(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # TODO 1: nếu nums[mid] > nums[hi] thì ...
        # TODO 2: nếu nums[mid] < nums[hi] thì ...
        # TODO 3: nếu bằng nhau thì ... (chỉ bỏ 1 phần tử, phía nào?)
        pass
    return nums[lo]

# Tự làm xong rồi mới mở test:
# assert exercise_find_min([1, 3, 5]) == 1
# assert exercise_find_min([2, 2, 2, 0, 1]) == 0
# assert exercise_find_min([1, 1, 1, 0, 1, 1]) == 0
# assert exercise_find_min([3, 3, 1, 3]) == 1

### Bài 4 — Tự viết lại: chia để trị (cách 5)

Không nhìn mục 12, viết lại hàm đệ quy. Nhớ hai điều: (1) base case, (2) phép cắt tỉa "đoạn đã sắp xếp thì trả đầu đoạn". Sau đó tự trả lời: vì sao khi `nums[lo] == nums[hi]` ta **không được** cắt tỉa? (Thử với `[1,0,1]`.)

In [35]:
def exercise_find_min_divide(nums: List[int]) -> int:
    def solve(lo, hi):
        # TODO 1: base case lo == hi
        # TODO 2: cắt tỉa: nếu nums[lo] < nums[hi] -> đoạn đã sắp xếp
        # TODO 3: chia đôi, gọi đệ quy hai nửa, lấy min
        raise NotImplementedError

    return solve(0, len(nums) - 1)

# Tự làm xong rồi mới mở test:
# assert exercise_find_min_divide([2, 2, 2, 0, 1]) == 0
# assert exercise_find_min_divide([1, 0, 1]) == 0
# assert exercise_find_min_divide([4, 5, 6, 7, 1, 3, 3, 3]) == 1

### Bài 5 — Biến thể: LeetCode 81 — Search in Rotated Sorted Array II

Cho mảng xoay **có thể trùng** `nums` và số `target`, trả về `True` nếu `target` có trong mảng, ngược lại `False`. Cố gắng dùng binary search.

```python
search([2,5,6,0,0,1,2], 0) -> True
search([2,5,6,0,0,1,2], 3) -> False
search([1,0,1,1,1], 0)     -> True
```

Ý tưởng chung: ở mỗi bước, một trong hai nửa `[lo..mid]` hoặc `[mid..hi]` là đoạn **đã sắp xếp**; kiểm tra `target` có nằm trong khoảng giá trị của nửa đó không để quyết định đi nửa nào. Nhưng khi `nums[lo] == nums[mid] == nums[hi]`, bạn **không biết** nửa nào đã sắp xếp — làm gì?

In [36]:
def exercise_search_rotated_dup(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    # TODO: while lo <= hi:
    #   - mid; nếu nums[mid] == target -> True
    #   - nếu nums[lo] == nums[mid] == nums[hi] -> thu hẹp cả hai đầu
    #   - elif nửa trái [lo..mid] đã sắp xếp -> target có trong [nums[lo], nums[mid]) không?
    #   - else nửa phải [mid..hi] đã sắp xếp -> target có trong (nums[mid], nums[hi]] không?
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_search_rotated_dup([2, 5, 6, 0, 0, 1, 2], 0) is True
# assert exercise_search_rotated_dup([2, 5, 6, 0, 0, 1, 2], 3) is False
# assert exercise_search_rotated_dup([1, 0, 1, 1, 1], 0) is True

### Bài 6 — Mini challenge: vị trí chỗ gãy khi có trùng

Viết hàm trả về **chỉ số nhỏ nhất `i`** sao cho `nums[i:] + nums[:i]` là mảng không giảm (tức là vị trí bắt đầu của phần B; nếu mảng đã sắp xếp thì trả `0`). Yêu cầu: dựa trên cách 4, vẫn đạt `O(log n)` khi ít trùng.

```python
pivot_index([4,5,6,7,0,1,4]) -> 4
pivot_index([1,1,1,2,1])     -> 4     # cách 4 nguyên bản cho lo = 0: sai chỉ số!
pivot_index([2,2,2,0,1])     -> 3
pivot_index([1,1,1])         -> 0
pivot_index([1,2,3])         -> 0
```

Câu hỏi then chốt: ở nhánh `==`, trước khi `hi -= 1`, làm sao biết `hi` có đang đứng **đúng** ở chỗ gãy không?

In [37]:
def exercise_pivot_index(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    # TODO: giống find_min_binary, nhưng ở nhánh nums[mid] == nums[hi]:
    #       nếu hi đang là chỗ gãy (so nums[hi - 1] với nums[hi]) thì trả hi ngay
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_pivot_index([4, 5, 6, 7, 0, 1, 4]) == 4
# assert exercise_pivot_index([1, 1, 1, 2, 1]) == 4
# assert exercise_pivot_index([2, 2, 2, 0, 1]) == 3
# assert exercise_pivot_index([1, 1, 1]) == 0
# assert exercise_pivot_index([1, 2, 3]) == 0

## 20. Gợi ý theo mức

**Bài 1**
- Mức 1: làm y như bảng trace ở mục 10; nhớ `mid = (lo + hi) // 2` lệch trái.
- Mức 2: (a) bước 1: `lo = 0`, `hi = 6`, `mid = 3`, `nums[3] = 0` so với `nums[6] = 2`.
- Mức 3: (a) nhánh lần lượt là `<`, `>`, `==`; (b) bước 1 là `3 == 3`.

**Bài 2**
- Mức 1: (a) xem lại mục 9; (b) toàn bằng nhau thì mỗi bước rơi vào nhánh nào?
- Mức 2: (b) mỗi bước `hi` giảm 1, dừng khi `hi == lo == 0`; (c) mảng không trùng → khoảng `log2(7)`.
- Mức 3: (b) `8 - 1`; (c) tự trace: `mid = 3` (`1 < 4`) rồi...

**Bài 3**
- Mức 1: ba nhánh tương ứng ba dòng trong bảng ở đầu mục 10.
- Mức 2: nhánh `>` bỏ cả `mid`; nhánh `<` giữ `mid`.
- Mức 3: `lo = mid + 1` / `hi = mid` / `hi -= 1`.

**Bài 4**
- Mức 1: base case trả `nums[lo]`.
- Mức 2: `if lo == hi or nums[lo] < nums[hi]: return nums[lo]`.
- Mức 3: `[1,0,1]` có `nums[0] == nums[2]` nhưng **không** sắp xếp; nếu cắt tỉa khi `<=` sẽ trả `1`.

**Bài 5**
- Mức 1: nếu `nums[lo] <= nums[mid]` (và không phải cả ba bằng nhau) thì nửa trái đã sắp xếp.
- Mức 2: ba số bằng nhau → `lo += 1; hi -= 1` (cả hai đầu đều bằng `nums[mid]`, mà `nums[mid] != target`, nên bỏ hai đầu không mất `target`).
- Mức 3: nửa trái sắp xếp: `if nums[lo] <= target < nums[mid]: hi = mid - 1 else: lo = mid + 1`; nửa phải tương tự với `nums[mid] < target <= nums[hi]`.

**Bài 6**
- Mức 1: chỗ gãy tại `i` nghĩa là `nums[i - 1] > nums[i]`.
- Mức 2: ở nhánh `==`, nếu `hi > 0` và `nums[hi - 1] > nums[hi]` thì `hi` chính là chỗ gãy → `return hi`.
- Mức 3: nhánh `>` và `<` giữ nguyên như cách 4; vì sao `lo` cuối cùng đúng? Nhánh `>` chỉ xảy ra khi chỗ gãy thật sự nằm trong `(mid, hi]`; nhánh `<` cho biết không có chỗ gãy trong `(mid, hi]`.

## 21. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1 (a)** `[2,2,0,0,1,2,2]` (là `[0,0,1,2,2,2,2]` xoay 2 lần):

| bước | lo | hi | mid | nums[mid] | nums[hi] | nhánh | quyết định |
|---:|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 6 | 3 | 0 | 2 | `<` | `hi = 3` |
| 2 | 0 | 3 | 1 | 2 | 0 | `>` | `lo = 2` |
| 3 | 2 | 3 | 2 | 0 | 0 | `==` | `hi = 2` |
| — | 2 | 2 | | | | dừng | trả `nums[2] = 0` |

Dừng sau **3 bước**, đi qua **đủ** ba nhánh.

**Bài 1 (b)** `[3,3,1,3]`:

| bước | lo | hi | mid | nums[mid] | nums[hi] | nhánh | quyết định |
|---:|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 3 | 1 | 3 | 3 | `==` | `hi = 2` |
| 2 | 0 | 2 | 1 | 3 | 1 | `>` | `lo = 2` |
| — | 2 | 2 | | | | dừng | trả `nums[2] = 1` |

Dừng sau **2 bước**. So với cách 3 (mục 9), chỉ khác bước 1: `hi = 2` thay vì `hi = 1` — và chính khác biệt đó cứu được số `1`.

**Bài 2:** (a) `1` (sai, đáp án đúng là `0`); (b) `7`; (c) `3` (`mid=3: 1<4 → hi=3`, `mid=1: 6>1 → lo=2`, `mid=2: 7>1 → lo=3`).

**Bài 4 — câu hỏi tư duy:** `nums[lo] == nums[hi]` không loại trừ được chỗ gãy, ví dụ `[1,0,1]` có `nums[0] == nums[2] == 1` nhưng min là `0` ở giữa. Chỉ `nums[lo] < nums[hi]` (nhỏ hơn **hẳn**) mới đảm bảo đoạn đã sắp xếp.

**Bài 6 — vì sao đúng:** nhánh `==` chỉ bỏ `hi` khi `hi` **không** phải chỗ gãy, nên chỗ gãy (nếu có) không bao giờ bị loại khỏi `[lo..hi]`. Nếu mảng không có chỗ gãy (đã sắp xếp), nhánh `>` không bao giờ xảy ra nên `lo` đứng yên ở `0`.

Các cell dưới kiểm chứng bài 1, bài 2 bằng máy, rồi đến đáp án code bài 3–6.

In [38]:
# Kiểm chứng Bài 1 và Bài 2
_ = trace_find_min_binary([2, 2, 0, 0, 1, 2, 2], show_window=False)
_ = trace_find_min_binary([3, 3, 1, 3], show_window=False)
print("Bài 2 (a):", find_min_no_dup([1, 1, 1, 0, 1, 1]))
print("Bài 2 (b):", count_steps([7] * 8))
print("Bài 2 (c):", count_steps([5, 6, 7, 1, 2, 3, 4]))

nums = [2, 2, 0, 0, 1, 2, 2]
bước 1: lo=0, hi=6, mid=3 | 0 < 2  -> hi = 3
bước 2: lo=0, hi=3, mid=1 | 2 > 0  -> lo = 2
bước 3: lo=2, hi=3, mid=2 | 0 == 0 -> hi = 2
dừng sau 3 bước: lo = hi = 2 -> min = 0
nums = [3, 3, 1, 3]
bước 1: lo=0, hi=3, mid=1 | 3 == 3 -> hi = 2
bước 2: lo=0, hi=2, mid=1 | 3 > 1  -> lo = 2
dừng sau 2 bước: lo = hi = 2 -> min = 1
Bài 2 (a): 1
Bài 2 (b): 7
Bài 2 (c): 3


In [39]:
def answer_find_min(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        elif nums[mid] < nums[hi]:
            hi = mid
        else:
            hi -= 1
    return nums[lo]


def answer_find_min_divide(nums: List[int]) -> int:
    def solve(lo, hi):
        if lo == hi or nums[lo] < nums[hi]:
            return nums[lo]
        mid = (lo + hi) // 2
        return min(solve(lo, mid), solve(mid + 1, hi))

    return solve(0, len(nums) - 1)


assert answer_find_min([1, 3, 5]) == 1
assert answer_find_min([2, 2, 2, 0, 1]) == 0
assert answer_find_min([1, 1, 1, 0, 1, 1]) == 0
assert answer_find_min([3, 3, 1, 3]) == 1
assert answer_find_min_divide([2, 2, 2, 0, 1]) == 0
assert answer_find_min_divide([1, 0, 1]) == 0
assert answer_find_min_divide([4, 5, 6, 7, 1, 3, 3, 3]) == 1
print("Đáp án bài 3, bài 4 PASS")

Đáp án bài 3, bài 4 PASS


In [40]:
def answer_search_rotated_dup(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:      # không biết nửa nào đã sắp xếp
            lo += 1
            hi -= 1
        elif nums[lo] <= nums[mid]:                # nửa trái [lo..mid] đã sắp xếp
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                      # nửa phải [mid..hi] đã sắp xếp
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False


def answer_pivot_index(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        elif nums[mid] < nums[hi]:
            hi = mid
        else:
            if nums[hi - 1] > nums[hi]:            # hi đứng đúng chỗ gãy (hi > mid >= 0 nên hi-1 hợp lệ)
                return hi
            hi -= 1
    return lo


def pivot_bruteforce(nums):
    n = len(nums)
    for i in range(n):
        r = nums[i:] + nums[:i]
        if all(r[j] <= r[j + 1] for j in range(n - 1)):
            return i


assert answer_search_rotated_dup([2, 5, 6, 0, 0, 1, 2], 0) is True
assert answer_search_rotated_dup([2, 5, 6, 0, 0, 1, 2], 3) is False
assert answer_search_rotated_dup([1, 0, 1, 1, 1], 0) is True
assert answer_pivot_index([4, 5, 6, 7, 0, 1, 4]) == 4
assert answer_pivot_index([1, 1, 1, 2, 1]) == 4
assert answer_pivot_index([2, 2, 2, 0, 1]) == 3
assert answer_pivot_index([1, 1, 1]) == 0
assert answer_pivot_index([1, 2, 3]) == 0

# Đối chiếu ngẫu nhiên với brute force
random.seed(81)
for _ in range(3000):
    nums = random_rotated(random.randint(1, 10), 0, 3)
    assert answer_find_min(nums) == min(nums)
    assert answer_find_min_divide(nums) == min(nums)
    assert answer_pivot_index(nums) == pivot_bruteforce(nums), nums
    for target in range(-1, 5):
        assert answer_search_rotated_dup(nums, target) == (target in nums), (nums, target)
print("Đáp án bài 5, bài 6 PASS + 3000 mảng ngẫu nhiên khớp brute force!")

Đáp án bài 5, bài 6 PASS + 3000 mảng ngẫu nhiên khớp brute force!


## 22. Template ngắn nên nhớ

```python
class Solution:
    def findMin(self, nums: List[int]) -> int:
        lo, hi = 0, len(nums) - 1
        while lo < hi:                    # dừng khi còn 1 ứng viên
            mid = (lo + hi) // 2
            if nums[mid] > nums[hi]:      # gãy ở bên phải mid  -> bỏ [lo..mid]
                lo = mid + 1
            elif nums[mid] < nums[hi]:    # [mid..hi] đi lên    -> bỏ (mid..hi]
                hi = mid
            else:                         # bằng nhau: bỏ đúng 1 phần tử (hi)
                hi -= 1
        return nums[lo]
```

Câu thần chú: **"So với đầu PHẢI. Lớn hơn thì nhảy qua mid, nhỏ hơn thì giữ mid, bằng nhau thì lùi hi một bước — vì giá trị ở hi vẫn còn bản sao ở mid."**

Và câu trả lời Follow up: **"Có trùng thì worst case `O(n)`, vì khi `nums[mid] == nums[hi]` không biết min nằm nửa nào; không thuật toán nào tránh được điều đó."**

## 23. Checklist trước khi submit

- So `nums[mid]` với `nums[hi]` (đầu **phải**), không phải `nums[lo]`?
- Nhánh `>`: `lo = mid + 1`; nhánh `<`: `hi = mid` (không phải `mid - 1`)?
- Có nhánh `==` riêng với `hi -= 1` (không phải `lo += 1`, không gộp vào `hi = mid`)?
- Vòng lặp là `while lo < hi` (không có dấu `=`)?
- Trả về **giá trị** `nums[lo]`, không phải chỉ số `lo`?
- Đã thử: hai example của đề, `n = 1`, `n = 2`, mảng chưa xoay `[1,2,3]`, toàn bằng nhau `[2,2,2]`, hai đầu bằng nhau `[1,1,1,0,1,1]` và `[1,0,1,1,1,1]`, `[3,3,1,3]`?
- Trả lời được câu Follow up: vì sao có trùng thì worst case `O(n)`?